# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muzammilsharf/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: "The Anatomy of Growing Content"

**Claim:** Pages trending upward (74,187) are 37.6% longer and 20% younger than pages trending downward (45,272); the paper reports this as a large-sample, directionally robust comparison.

**Where the label comes from:** "up" vs "down" is a 30-day trend-direction bucket, the same kind of proxy label our own model uses. The paper is upfront that this is observational, not causal.

**My methodology question:** the paper's own Limitations section separately states "content age confounds model-performance comparisons," but Finding 1 reports word count and age as two independent differences between growing and declining content without addressing whether they're actually confounded with each other, older pages might simply have been written in an earlier, shorter-content era of the site, rather than age and length being two separate causal levers. Does the analysis (or a follow-up) hold age constant while comparing word count, or vice versa, to check whether one of these two "differences" is actually just a proxy for the other? This doesn't undercut the finding, it's exactly the kind of check that would make an already-strong sample-size result even more defensible.

### Finding 2: "The Freshness Multiplier"

**Claim:** 365+ day content refreshed within 30 days shows a 3.2x health-score boost and 57x more impressions, framed as "one of the strongest measured levers available."

**Where the label comes from:** health score is the paper's own composite metric (impressions + position + CTR + scroll depth), and "refreshed" is presumably a binary flag on whether an edit happened in the last 30 days.

**My methodology question:** this compares refreshed vs. non-refreshed pages within the 365+ bucket, but doesn't say whether refresh timing was randomly assigned or editor-selected. If editors tend to refresh pages they already judge to have recovery potential (existing backlinks, a topic still in demand), the 57x impression gap could partly reflect that selection, not the refresh action itself. The paper is careful elsewhere (it explicitly flags the 361+ bucket's 283:1 ratio as unstable, "only 1 declining page"), so this isn't a paper that hides its caveats, this specific comparison just doesn't state whether refresh assignment was closer to random or closer to editor-chosen, and that distinction changes how strongly "refresh timing is one of the strongest measured levers" should be read.

**Constructive note, tying back to our own work:** both questions above are versions of a check we had to run on our own model this week, whether an apparent driver (age, refresh timing) is really doing the work, or is a stand-in for something else nearby in time. Our own `content_age_days` feature turned out to be a disguised calendar-month signal rather than genuine per-page aging, worth naming here since it's the same category of question, not a "gotcha" specific to this paper.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
# BEFORE: naive split (content-level grouping, which we later found still leaks via same-client similarity)
gss_before = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss_before.split(model_df_encoded, groups=model_df_encoded["content_hash_id"]))
tr_b, te_b = model_df_encoded.iloc[tr_idx], model_df_encoded.iloc[te_idx]

gbm_before = lgb.LGBMClassifier(n_estimators=300, max_depth=6, num_leaves=31,
                                  learning_rate=0.05, is_unbalance=True, random_state=42, verbosity=-1)
gbm_before.fit(tr_b[features], tr_b["is_declining_label"])
te_b = te_b.copy()
te_b["risk"] = gbm_before.predict_proba(te_b[features])[:, 1]
p50_before = te_b.nlargest(50, "risk")["is_declining_label"].mean()

# AFTER: client-level holdout, averaged across 5 seeds (the honest split)
print(f"BEFORE (content-grouped split): Precision@50 = {p50_before:.3f}")
print(f"AFTER (client-holdout, mean of 5 seeds): Precision@50 = {np.mean(scores):.3f} (σ={np.std(scores):.3f})")

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# Deliberate leak, same demonstration as w03, run on the FINAL feature set this time
leak_test = model_df_encoded.copy()
leak_test["leaky_feature"] = leak_test["is_declining_label"]

leaky_features = features + ["leaky_feature"]
tr_l, te_l = leak_test.iloc[train_idx], leak_test.iloc[test_idx]

rf_leaky = RandomForestClassifier(n_estimators=50, random_state=42).fit(tr_l[leaky_features].fillna(0), tr_l["is_declining_label"])
rf_honest = RandomForestClassifier(n_estimators=50, random_state=42).fit(tr_l[features].fillna(0), tr_l["is_declining_label"])

print(f"Accuracy WITH leaked feature: {rf_leaky.score(te_l[leaky_features].fillna(0), te_l['is_declining_label']):.3f}")
print(f"Accuracy WITHOUT (honest): {rf_honest.score(te_l[features].fillna(0), te_l['is_declining_label']):.3f}")

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original (too bold): "Our model predicts which pages will lose search traffic."

Rewritten (safe): "Under a client-holdout validation design, the model's top-50 ranked pages showed an average Precision@50 of 78.8% (σ=0.124 across 5 random splits) for identifying pages with observed declining search impressions in the 30 days following the feature window. This is a decision-support ranking, not a guarantee, individual predictions should be reviewed by a human before action is taken, and performance during anomalous periods (e.g. the March-April 2026 spike, excluded from this evaluation) is not yet established."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.